# ME-TST / GLSD 环境精简版

本 notebook 只保留原 `GLSD+ME-TST.ipynb` 中**最终成功建立运行环境所需的步骤**，
删除前面反复失败的 Python 3.10 / requirements / conda 尝试，以及错误的 dlib patch。

用途：
- 恢复 ME-TST 原工程；
- 建立原实验最终使用的 Python 3.8 + PyTorch 2.1.0 环境；
- 安装原 notebook 最终成功的依赖版本；
- 验证 `Utils`、`mamba_ssm`、`causal_conv1d` 可导入。

**不会运行 backbone、不会生成 response、不会修改 Drive 中 locked evidence。**

建议：Colab 运行时选择 GPU，然后从上到下依次执行。


## 1. 挂载 Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("DRIVE_MOUNT = PASS")


## 2. 干净恢复 ME-TST 仓库

这里会删除 `/content/ME-TST` 的**临时 Colab 目录**后重新 clone。
不会删除 Google Drive 中的任何文件。
这样可以避免之前残留的不完整仓库导致 `Utils` 缺失。


In [ ]:
%cd /content
!rm -rf /content/ME-TST
!git clone https://github.com/zizheng-guo/ME-TST.git /content/ME-TST

from pathlib import Path

repo = Path("/content/ME-TST")
assert (repo / "Utils").exists(), "ME-TST/Utils not found after clean clone"
assert (repo / "Utils/mean_average_precision_str").exists(), "mean_average_precision_str folder missing"

print("METST_REPO_CLONE = PASS")
print("repo =", repo)


## 3. 安装 micromamba


In [ ]:
%cd /content
!rm -rf /content/bin
!wget -qO- https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj

from pathlib import Path
mamba = Path("/content/bin/micromamba")
assert mamba.exists(), "micromamba binary missing"
print("MICROMAMBA_INSTALL = PASS")
print("micromamba =", mamba)


## 4. 创建最终成功的 ME-TST 环境

对应原 notebook 最终采用的环境：
- Python 3.8
- PyTorch 2.1.0
- torchvision 0.16.0
- torchaudio 2.1.0
- CUDA 12.1


In [ ]:
!/content/bin/micromamba env remove -n ME-TST -y || true

!/content/bin/micromamba create -y -n ME-TST   -c pytorch -c nvidia -c conda-forge   python=3.8   pytorch=2.1.0   torchvision=0.16.0   torchaudio=2.1.0   pytorch-cuda=12.1   pip

!/content/bin/micromamba run -n ME-TST python --version


## 5. 验证 PyTorch / CUDA


In [ ]:
!/content/bin/micromamba run -n ME-TST python -c "import torch; print('torch =', torch.__version__); print('torch cuda =', torch.version.cuda); print('cuda available =', torch.cuda.is_available())"


## 6. 安装原 notebook 最终成功的基础依赖


In [ ]:
%cd /content/ME-TST

!/content/bin/micromamba run -n ME-TST pip install   natsort==7.1.1   einops==0.8.0   tqdm==4.66.4   pandas==1.1.3   scikit-learn==0.23.2   scipy==1.5.4   xlrd==1.2.0   opencv-python-headless==4.5.1.48

!/content/bin/micromamba run -n ME-TST pip install --no-deps timm==1.0.7


## 7. 验证基础依赖与 ME-TST Utils


In [ ]:
%cd /content/ME-TST

!/content/bin/micromamba run -n ME-TST python -c "import torch,natsort,einops,timm,pandas,sklearn,scipy,cv2; print('torch =',torch.__version__); print('cuda =',torch.cuda.is_available()); print('BASIC_OK')"

!/content/bin/micromamba run -n ME-TST python -c "from Utils.mean_average_precision_str.mean_average_precision import MeanAveragePrecision2d; print('METST_UTILS_IMPORT = PASS')"


## 8. 安装 causal-conv1d


In [ ]:
%cd /content/ME-TST

!/content/bin/micromamba run -n ME-TST pip install     causal-conv1d==1.4.0     --no-build-isolation


## 9. 安装 mamba-ssm


In [ ]:
%cd /content/ME-TST

!/content/bin/micromamba run -n ME-TST pip install     mamba-ssm==1.2.0.post1     --no-build-isolation


## 10. 最终环境检查


In [ ]:
%cd /content/ME-TST

!/content/bin/micromamba run -n ME-TST python -c "import sys, torch, pandas, mamba_ssm, causal_conv1d; from Utils.mean_average_precision_str.mean_average_precision import MeanAveragePrecision2d; print('Python:', sys.version); print('torch:', torch.__version__); print('CUDA:', torch.cuda.is_available()); print('pandas:', pandas.__version__); print('mamba_ssm: OK'); print('causal_conv1d: OK'); print('Utils metric: OK'); print('ME_TST_ENVIRONMENT = PASS')"


## 下一步

当最后一格输出 `ME_TST_ENVIRONMENT = PASS` 后，环境恢复完成。

**不要重新生成 official response。**
本阶段只读取已经锁定的 ME-TST+ official dumps / evidence，
固定 a0=2.0、rho=1.0，测试 generalized mean 的 p=1/2/4/8；
p=inf 只做诊断，不进入正式联合选择。


In [ ]:
from google.colab import drive, files
from pathlib import Path
import zipfile, io, subprocess, sys
drive.mount('/content/drive')
uploaded = files.upload()
name = 'metst_generalized_mean_screening.zip'
assert name in uploaded, '请上传 metst_generalized_mean_screening.zip'
task_dir = Path('/content/glsd_generalized_mean_screening')
task_dir.mkdir(exist_ok=True)
allowed = {'run_generalized_mean_screening.py', 'official_response_component_ablation.py', 'test_generalized_mean_screening.py', 'README.md'}
with zipfile.ZipFile(io.BytesIO(uploaded[name])) as archive:
    assert set(archive.namelist()) == allowed, '脚本包内容与预期不一致'
    archive.extractall(task_dir)
print('第一阶段 generalized-mean 脚本已准备:', task_dir)


In [ ]:
test = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(task_dir), '-p', 'test_generalized_mean_screening.py', '-v'], check=True)
print('GENERALIZED_MEAN_UNIT_TESTS = PASS')
subprocess.run([sys.executable, str(task_dir / 'run_generalized_mean_screening.py'),
                '--setting', 'both', '--check-inputs'], check=True)


In [ ]:
from datetime import datetime, timezone
import os
run_tag = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
output_base = Path('/content/drive/MyDrive/GLSD_GENERALIZED_MEAN_SCREENING')
output_base.mkdir(parents=True, exist_ok=True)
screening_output = output_base / ('screen_' + run_tag)
screening_log = output_base / ('screen_' + run_tag + '.log')
bootstrap = """
import runpy, sys
import pandas as pd

if not hasattr(pd.DataFrame, "append"):
    def dataframe_append(
        self, other, ignore_index=False,
        verify_integrity=False, sort=False
    ):
        return pd.concat(
            [self, other],
            ignore_index=ignore_index,
            verify_integrity=verify_integrity,
            sort=sort,
        )
    pd.DataFrame.append = dataframe_append

sys.argv = sys.argv[1:]
runpy.run_path(sys.argv[0], run_name="__main__")
"""

command = [
    sys.executable, "-u", "-c", bootstrap,
    str(task_dir / "run_generalized_mean_screening.py"),
    "--setting", "both",
    "--output", str(screening_output),
]
env = dict(os.environ)
env['PYTHONPATH'] = os.pathsep.join(dict.fromkeys([os.getcwd()] + [p for p in sys.path if p] + [env.get('PYTHONPATH', '')]))
print('输出目录:', screening_output)
print('完整日志:', screening_log)
with screening_log.open('x', encoding='utf-8') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    try:
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        returncode = process.wait()
    except BaseException:
        process.terminate()
        raise
assert returncode == 0, f'运行停止，请查看日志: {screening_log}'
assert (screening_output / 'completion.json').is_file(), '缺少完整完成标志'
